# Brief 01, corrigé

In [1]:
import os
from pathlib import Path
import subprocess
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values 

> ⚠️ Il faut faire attention à bien modifier la variable `RELEVES_ROOT` en fonction de l'architecture de votre dossier / repository.

In [2]:
RELEVES_ROOT = "releves"
# On ne sélectionne que les dossier grâce à l'usage de `.is_dir()` 
SITES = sorted(d.name for d in Path(RELEVES_ROOT).iterdir() if d.is_dir())
print(SITES)

['animalis', 'bitiba_fr', 'chronovet', 'clubvetshop', 'maxizoo', 'pharmacy4pets', 'univers_veto', 'vetoplus', 'vetostore', 'zooplus_fr']


In [3]:
# ⚠️ Il faut faire attention à bien modifier la variable DATABASE_URL, avec vos valeurs
conn = psycopg2.connect(
    dbname="vetprice",
    user="postgres",
    password="Mkilo1990",
    host="localhost",
    port=5432
)
conn.autocommit = False
cur = conn.cursor()

## 1. Profilage et modélisation

### Organisation des données

Chaque site a été scrapé plusieurs fois : 
- une fois en avril
- deux en juillet.
-
Un "run" (i.e. une exécution d'un scraper pour un site) produit un dossier horodaté `AAAA-MM-JJ_HHMM`, qui contient les produits collectés ce jour-là. Rien n'est écrasé d'un run à l'autre : 🔥🔥🔥 **c'est ce qui rend l'historisation possible**.

```
releves/
├── chronovet/
│   ├── 2026-04-01_1919/          run du 1er avril à 19h19
│   ├── 2026-07-12_1150/          run du 12 juillet à 11h50  (partiel car interrompu)
│   ├── 2026-07-12_1242/          run du 12 juillet à 12h42  (complet)
│   └── 2026-07-18_1334/          run du 18 juillet à 13h34
├── univers_veto/
│   ├── 2026-04-01_1847/
│   ├── 2026-04-01_1918/
│   ├── 2026-07-12_1148/
│   ├── 2026-07-12_1150/
│   ├── 2026-07-12_1242/
│   ├── 2026-07-18_1334/
│   └── 2026-07-18_1355/
├── animalis/    …
├── bitiba_fr/   …
├── clubvetshop/ …
├── maxizoo/     …
├── pharmacy4pets/ …
├── vetoplus/    …
├── vetostore/   …
└── zooplus_fr/  …
```

Un run contient : 

```
2026-07-18_1334/
├── products.jsonl    un produit par ligne, au format JSON (le champ scraped_at porte l'instant de collecte)
└── meta.json         pas toujours présent, métadonnées du run (products_scraped, http_requests, http_errors, completed, resumed)
```

**Pourquoi y a-t-il plusieurs dossiers le même jour ?**

- Un run peut être interrompu (site lent, coupure, mise en veille de l'ordinateur portable) puis relancé.
- Chaque relance (i.e. nouveau run) crée un nouveau dossier horodaté (i.e. le dossier du run).
- Par exemple, sur chronovet le 12 juillet, `_1150` est un run partiel et `_1242` un run complet du même jour.
- Le champ `HHMM` du nom permet de savoir lequel est le dernier


## Nombre de fichiers de relevés (i.e. un fichier par run)

In [10]:
RELEVES = [
    (site, run_dir / "products.jsonl")
    for site in SITES
    for run_dir in sorted((Path(RELEVES_ROOT) / site).iterdir())
    if (run_dir / "products.jsonl").exists()
]
print(len(RELEVES), "relevés (fichiers) au total")

41 relevés (fichiers) au total


## Profiler tous les fichiers

In [6]:
# Profilage sur TOUS les fichiers : chaque run de chaque site
def profiler_tous_les_fichiers():
    rows = []
    for site in SITES:
        for run_dir in (Path(RELEVES_ROOT) / site).iterdir():
            f = run_dir / "products.jsonl"
            # S'il n'y a pas de fichier pour ce dossier de run
            # on passe au `run_dir_ suivant dans la boucle (avec `continue`)
            if not f.exists():
                continue
            df = pd.read_json(f, lines=True, dtype=False)
            n = len(df)
            # # ean absent : soit colonne manquante, soit valeur vide (ou NaN, donc à bien vérifier)
            if "ean" in df.columns:
                ean = df["ean"]
                # Ci-dessous, ce n'est pas la façon la plus propre, 
                # mais cette écriture permet d'éviter une étape préliminaire de cleaning
                a_ean = ean.notna() & (ean.astype(str).str.strip() != "")
            else:
                ean, a_ean = pd.Series([None] * n, dtype=object), pd.Series([False] * n)
            rows.append({
                "site": site,
                "run": run_dir.name,
                "lignes": n,
                "sans_ean": int((~a_ean).sum()),
                "doublons_ean": int(ean[a_ean].duplicated().sum()),  # doublons internes au run
                "colonnes": df.shape[1],
            })
    return pd.DataFrame(rows)

In [7]:
stats = profiler_tous_les_fichiers()

# 💫 On ajoute le pourcentage de lignes sans ean par run
stats["sans_ean_%"] = (100 * stats["sans_ean"] / stats["lignes"]).round(1)
print(stats.to_string(index=False))

         site             run  lignes  sans_ean  doublons_ean  colonnes  sans_ean_%
     animalis 2026-04-01_1919    2866         4           634        17         0.1
     animalis 2026-07-12_1150     775         0           148        17         0.0
     animalis 2026-07-12_1242   10782         7          3434        17         0.1
     animalis 2026-07-18_1334   53561       820         31114        17         1.5
    bitiba_fr 2026-04-01_1921   21008         0          8062        20         0.0
    bitiba_fr 2026-07-12_1242   12763         0             0        12         0.0
    bitiba_fr 2026-07-18_1334   12659         0             0        12         0.0
    chronovet 2026-04-01_1919    1631         0             0        14         0.0
    chronovet 2026-07-12_1150      90         0             0        19         0.0
    chronovet 2026-07-12_1242    2436         5             0        19         0.2
    chronovet 2026-07-18_1334    2453         5             0        19     

## Agrégat par site (sur tous les runs)

In [8]:
agg = stats.groupby("site")[["lignes", "sans_ean", "doublons_ean"]].sum()
agg

,lignes,sans_ean,doublons_ean
site,,,
animalis,67984,831,35330
bitiba_fr,46430,0,8062
chronovet,6610,10,0
clubvetshop,18524,10581,8
maxizoo,18675,1017,1
pharmacy4pets,4120,4120,0
univers_veto,2146,135,0
vetoplus,7367,643,147
vetostore,10585,10585,0


## On calcule le pourcentage total de lignes sans ean

In [9]:
profils = {
    s: (int(r.lignes), r.sans_ean / r.lignes, int(r.doublons_ean)) 
    for s, r in agg.iterrows()
}


print(f"\nTotal : {int(stats['lignes'].sum())} lignes sur {len(stats)} fichiers, "
      f"{100 * agg['sans_ean'].sum() / agg['lignes'].sum():.1f}% sans ean global")


Total : 388784 lignes sur 41 fichiers, 7.2% sans ean global


## Il y a 7.2 % d'ean manquants -> il faudra une clef de repli
- ✅ on crée donc immédiatement une variable clé que l'on créera avant l'insertion